In [ ]:
from vivarium import Artifact, InteractiveContext
import pandas as pd, numpy as np, os

In [ ]:
! pip list | grep vivarium

# [software verion + hash . date]

In [ ]:
! pip freeze | grep vivarium

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from vivarium import InteractiveContext, Artifact

In [ ]:
import vivarium_gates_mncnh
from vivarium.framework.configuration import build_model_specification
from pathlib import Path

from vivarium_gates_mncnh.constants.data_values import (
    SIMULATION_EVENT_NAMES,
    COLUMNS,
    PREGNANCY_OUTCOMES,
    PIPELINES,
)

path = Path(vivarium_gates_mncnh.__file__).parent / 'model_specifications/model_spec.yaml'
custom_model_specification = build_model_specification(path)
custom_model_specification.configuration.input_data.input_draw_number = 60

custom_model_specification.configuration.population.population_size = 20_000 * 3

included_components = custom_model_specification.components.vivarium_gates_mncnh.components
included_components

In [ ]:
artifact_path = custom_model_specification.configuration.input_data.artifact_path
art = Artifact(artifact_path)

artifact_path

In [ ]:
draw_num = custom_model_specification.configuration.input_data.input_draw_number
draw = 'draw_' + str(draw_num)
draw

CHECK: puerperal sepsis hemoglobin effect size in the artifact does not vary by sex, age, or year.

Suite: artifact tests.

Type: precise assert.

In [ ]:
sepsis_on_hemoglobin_shift = art.load("cause.maternal_sepsis_and_other_maternal_infections.hemoglobin_shift")[draw]
assert len(sepsis_on_hemoglobin_shift) == 2  # early and late postpartum
sepsis_on_hemoglobin_shift

In [ ]:
def check_hemoglobin_shift(sim: InteractiveContext):
    pop = sim.get_population([COLUMNS.MATERNAL_SEPSIS, COLUMNS.PREGNANCY_OUTCOME])
    # We model maternal sepsis on live and still births only
    live_births = pop.loc[
        (pop[COLUMNS.PREGNANCY_OUTCOME] == PREGNANCY_OUTCOMES.LIVE_BIRTH_OUTCOME) |
        (pop[COLUMNS.PREGNANCY_OUTCOME] == PREGNANCY_OUTCOMES.STILLBIRTH_OUTCOME) |
        (pop[COLUMNS.PREGNANCY_OUTCOME] == PREGNANCY_OUTCOMES.FULL_TERM_OUTCOME)
    ]
    has_sepsis = live_births[COLUMNS.MATERNAL_SEPSIS]
    sepsis_idx = live_births.index[has_sepsis]
    no_sepsis_idx = live_births.index[~has_sepsis]
    if len(sepsis_idx) == 0:
        return
    # Get hemoglobin exposure values from the pipeline
    hgb = sim.get_population(PIPELINES.HEMOGLOBIN_EXPOSURE).loc[live_births.index]
    mean_hgb_sepsis = hgb.loc[sepsis_idx].mean()
    mean_hgb_no_sepsis = hgb.loc[no_sepsis_idx].mean()

    return {'step': sim._clock.step_name, 'mean_hgb_sepsis': mean_hgb_sepsis, 'mean_hgb_no_sepsis': mean_hgb_no_sepsis}

In [ ]:
def run_all_steps(sim: InteractiveContext):
    step_shifts = []
    while sim.get_number_of_steps_remaining() > 0:
        step_shift = check_hemoglobin_shift(sim)
        if step_shift:
            step_shifts.append(step_shift)
        sim.step()
    return pd.DataFrame(step_shifts)

In [ ]:
def run_to_step(sim: InteractiveContext, step_name: str):
    while sim._clock.step_name != step_name:
        sim.step()
    return sim

In [ ]:
sim = InteractiveContext(custom_model_specification)
step_shifts = run_all_steps(sim)

In [ ]:
step_shifts["observed_diff"] = step_shifts["mean_hgb_sepsis"] - step_shifts["mean_hgb_no_sepsis"]
step_shifts

In [ ]:
step_shifts = step_shifts.set_index("step")[["observed_diff"]].diff().iloc[1:]
step_shifts

In [ ]:
assert np.isclose(step_shifts.loc["mortality"], 0)
assert np.isclose(step_shifts.loc["early_postpartum"], sepsis_on_hemoglobin_shift["early_postpartum"])
assert np.isclose(step_shifts.loc["late_postpartum"], sepsis_on_hemoglobin_shift["late_postpartum"] - sepsis_on_hemoglobin_shift["early_postpartum"])
assert np.isclose(step_shifts.loc["early_neonatal_mortality"], -sepsis_on_hemoglobin_shift["late_postpartum"])
assert np.isclose(step_shifts.loc["late_neonatal_mortality"], 0)